<a href="https://colab.research.google.com/github/nguyenduyvu61107/BTAINGUYENDUYVU2026/blob/main/Nh%E1%BA%ADn_di%E1%BB%87n_ch%E1%BB%89_tay_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics roboflow scikit-fuzzy opencv-python matplotlib
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from roboflow import Roboflow
from ultralytics import YOLO
import skfuzzy as fuzzy
from skfuzzy import control as ctrl
from google.colab import files
import tensorflow as tf
from google.colab import drive
import shutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 136.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 

In [ ]:

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from roboflow import Roboflow
from ultralytics import YOLO
import skfuzzy as fuzzy
from skfuzzy import control as ctrl
from google.colab import files
import tensorflow as tf
from google.colab import drive
import shutil
print(" Đang kết nối với Google Drive...")
drive.mount('/content/drive')
drive_model_path = '/content/drive/MyDrive/best_yolov8_palmistry.pt'
has_pretrained_model = False

if os.path.exists(drive_model_path):
    print(" Tìm thấy file mô hình 'best_yolov8_palmistry.pt' có sẵn trên My Drive")
    yolov8_model = YOLO(drive_model_path)
    print(" Đã nạp mô hình từ Google Drive thành công.")
    has_pretrained_model = True
else:
    print(" Không tìm thấy mô hình sẵn trên Drive. Hệ thống sẽ tiến hành tải data và huấn luyện mới...")

ROBOFLOW_API_KEY = "EZ6sCm39cKY3vS7G6L7k"

if not has_pretrained_model:

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("palmistry-ccmq5").project("palmistry-dhpnb")
    dataset = project.version(3).download("yolov8")

    model = YOLO('yolov8n.pt')
    device_to_use = 0 if tf.config.list_physical_devices('GPU') else 'cpu'

    model.train(
        data=f"{dataset.location}/data.yaml",
        epochs=50,
        imgsz=640,
        batch=16,
        device=device_to_use,
        workers=2,
        save=True,
        project="Palmistry_YOLOv8",
        name="train_7k_images"
    )

    local_best_weights = '/content/runs/detect/Palmistry_YOLOv8/train_7k_images/weights/best.pt'

    if os.path.exists(local_best_weights):
        shutil.copy(local_best_weights, drive_model_path)
        print(f" ĐÃ SAO CHÉP VÀ LƯU THÀNH CÔNG mô hình lên My Drive tại: {drive_model_path}")

        yolov8_model = YOLO(drive_model_path)
    else:
        print("Lỗi: Không tìm thấy file trọng số sau khi huấn luyện xong cục bộ.")

print("\n Huấn luyện hoàn tất!")

def setup_fuzzy_fortune_teller():

    tinh_duyen = ctrl.Antecedent(np.arange(0, 101, 1), 'tinh_duyen')
    sinh_menh = ctrl.Antecedent(np.arange(0, 101, 1), 'sinh_menh')
    ket_qua_boi = ctrl.Consequent(np.arange(0, 101, 1), 'ket_qua_boi')

    tinh_duyen['ngan'] = fuzzy.trimf(tinh_duyen.universe, [0, 0, 50])
    tinh_duyen['dai'] = fuzzy.trimf(tinh_duyen.universe, [40, 100, 100])

    sinh_menh['ngan'] = fuzzy.trimf(sinh_menh.universe, [0, 0, 50])
    sinh_menh['dai'] = fuzzy.trimf(sinh_menh.universe, [40, 100, 100])

    ket_qua_boi['doan_menh_hanh_phuc'] = fuzzy.trimf(ket_qua_boi.universe, [0, 25, 55])
    ket_qua_boi['vien_man_truong_tho'] = fuzzy.trimf(ket_qua_boi.universe, [45, 75, 100])

    rule1 = ctrl.Rule(tinh_duyen['dai'] & sinh_menh['ngan'], ket_qua_boi['doan_menh_hanh_phuc'])
    rule2 = ctrl.Rule(tinh_duyen['dai'] & sinh_menh['dai'], ket_qua_boi['vien_man_truong_tho'])

    boi_toan_control = ctrl.ControlSystem([rule1, rule2])
    system_simulation = ctrl.ControlSystemSimulation(boi_toan_control)

    return system_simulation

fuzzy_engine = setup_fuzzy_fortune_teller()
def predict_and_tell_fortune(image_path):

    results = yolov8_model(image_path)[0]

    img = cv2.imread(image_path)
    img_height, img_width, _ = img.shape

    color_map = {
        'head': (0, 0, 255),
        'heart': (255, 192, 203),
        'life': (0, 255, 0),
        'fate': (255, 255, 0)
    }

    line_lengths = {'head': 0, 'heart': 0, 'life': 0, 'fate': 0}

    for box in results.boxes:
        class_id = int(box.cls[0])
        class_name = results.names[class_id]
        xmin, ymin, xmax, ymax = map(int, box.xyxy[0].tolist())

        color = color_map.get(class_name, (255, 255, 255))

        cv2.rectangle(img, (xmin, ymin), (xmax, ymax), color, 3)

        cv2.putText(img, class_name.upper(), (xmin, ymin - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        length = ((xmax - xmin)**2 + (ymax - ymin)**2)**0.5
        line_lengths[class_name] = length

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 10))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()

    print("\n" + "="*80)
    print("🟦 Màu xanh dương: Học vấn | 🟪 Màu hồng: Tình duyên | 🟩 Màu xanh lá: Sinh mệnh | 🟨 Màu vàng: Vận mệnh")
    print("="*80)

    val_heart = min((line_lengths['heart'] / 300.0) * 100, 100)
    val_life = min((line_lengths['life'] / 300.0) * 100, 100)

    fuzzy_engine.input['tinh_duyen'] = val_heart
    fuzzy_engine.input['sinh_menh'] = val_life

    try:
        fuzzy_engine.compute()
        output_score = fuzzy_engine.output['ket_qua_boi']

        if output_score <= 50:
            print(f"Tiên đoán: Đường tình duyên sâu sắc, nồng nhiệt, tuy nhiên năng lượng sinh mệnh có phần suy giảm ở hậu vận. Cần chú ý giữ gìn sức khỏe, cân bằng cảm xúc tránh lao lực!")
        else:
            print(f"Tiên đoán: Số mệnh đại cát, tình duyên đong đầy trọn vẹn kết hợp với cung sinh mệnh vững vàng, cuộc sống trường thọ viên mãn về sau.")
    except Exception as m_error:
        print(" Hệ thống nhận diện hình dạng chỉ tay chưa đủ rõ nét. Vui lòng chụp lại ảnh rõ ràng, đủ ánh sáng hơn để quẻ bói chuẩn xác!")


print("Hãy chọn và tải lên 1 tấm ảnh bàn tay từ máy")
uploaded = files.upload()
for filename in uploaded.keys():
    print(f" Đang tiến hành phân tích bức ảnh '{filename}'...")
    predict_and_tell_fortune(filename)

🚀 Đang tiến hành tải tập dữ liệu từ Roboflow...
loading Roboflow workspace...
loading Roboflow project...
✅ Tải dataset thành công! Dữ liệu lưu tại: /content/palmistry-3
🏋️ Bắt đầu quá trình huấn luyện mô hình với ~7,000 ảnh...
🖥️ Thiết bị sẽ sử dụng để huấn luyện: 0
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/palmistry-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, k